# 🏭 Workshop — Dados Estruturados + Não Estruturados com IA
### Indústria Petroquímica — Gestão de Contratos, Fornecedores e Procedimentos

---

**Duração:** \~3 horas | **Módulos independentes** — execute na ordem ou pule para o módulo desejado.

| Módulo | Descrição | Tempo Estimado |
|--------|-----------|----------------|
| **Módulo 0** | Setup — Criar tabelas e volume com dados | 10 min |
| **Módulo 1** | Genie — Perguntas em linguagem natural sobre dados estruturados | 20 min |
| **Módulo 2** | AI Functions — Enriquecer dados com IA via SQL *(opcional)* | 20 min |
| **Módulo 3** | Unity Catalog Functions — Funções reutilizáveis para Genie e Agentes *(opcional)* | 20 min |
| **Módulo 4** | AgentBricks — Knowledge Assistant com PDFs (dados não estruturados) | 30 min |
| **Módulo 5** | Integração AgentBricks + Genie — Cruzar dados estruturados e não estruturados | 30 min |
| **Módulo 6** | AI/BI Dashboard — Visualização inteligente dos dados | 20 min |
| **Módulo 7** | Databricks Apps — Publicar o agente como aplicação | 20 min |

---

**Cenário do Workshop:**  
Você é um(a) engenheiro(a) de dados em uma empresa petroquímica brasileira que produz resinas termoplásticas (PE, PP, PVC).  
A empresa possui **contratos** com diversos **fornecedores** de serviços e materiais, **ordens de serviço** operacionais, além de **documentos PDF** de contratos e **procedimentos operacionais** de SMS, manutenção, qualidade e engenharia.

O objetivo é construir um **agente inteligente** que consiga responder perguntas cruzando esses dois mundos: os dados tabulares (quem fornece o quê, quanto custa, status) com os documentos não estruturados (cláusulas contratuais, procedimentos de segurança).

---
# Módulo 0 — Setup do Ambiente
> **Objetivo:** Criar as tabelas a partir dos CSVs e o volume para armazenar os PDFs.  
> **Pré-requisito:** Ter acesso de escrita ao catálogo e schema informados abaixo.

In [0]:
# ============================================================
# PARÂMETROS DO WORKSHOP - Preencha antes de executar
# ============================================================

# Catálogo e Schema onde as tabelas e volume serão criados
CATALOG = "workshop_caroljf"  # <-- Altere para o seu catálogo
SCHEMA = "workshop"  # <-- Altere para o seu schema

# Prefixo para nomear tabelas e volume (ex: "petro", "workshop", suas iniciais...)
PREFIX = "caroljf"  # <-- Altere se desejar

# Nome do volume (pode customizar)
VOLUME_NAME = f"{PREFIX}_volume"

# ============================================================
# Variáveis derivadas (não alterar)
# ============================================================
FULL_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"
TABLE_CONTRATOS = f"{CATALOG}.{SCHEMA}.{PREFIX}_contratos"
TABLE_FORNECEDORES = f"{CATALOG}.{SCHEMA}.{PREFIX}_fornecedores"
TABLE_ORDENS_SERVICO = f"{CATALOG}.{SCHEMA}.{PREFIX}_ordens_servico"

# ============================================================
# Widgets — necessários para que células SQL usem ${VAR}
# No Serverless, ${...} em SQL só resolve widgets, não variáveis Python.
# ============================================================
for name, value in {
    "CATALOG": CATALOG,
    "SCHEMA": SCHEMA,
    "PREFIX": PREFIX,
    "VOLUME_NAME": VOLUME_NAME,
    "TABLE_CONTRATOS": TABLE_CONTRATOS,
    "TABLE_FORNECEDORES": TABLE_FORNECEDORES,
    "TABLE_ORDENS_SERVICO": TABLE_ORDENS_SERVICO,
}.items():
    dbutils.widgets.text(name, value)

print("📋 Configuração do Workshop")
print("=" * 50)
print(f"Catálogo:           {CATALOG}")
print(f"Schema:             {SCHEMA}")
print(f"Prefixo:            {PREFIX}")
print(f"Volume:             {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
print(f"Volume Path:        {FULL_VOLUME_PATH}")
print(f"")
print(f"Tabelas:")
print(f"  • {TABLE_CONTRATOS}")
print(f"  • {TABLE_FORNECEDORES}")
print(f"  • {TABLE_ORDENS_SERVICO}")
print(f"\n✅ Widgets criados — células SQL podem usar ${{TABLE_CONTRATOS}}, etc.")

📋 Configuração do Workshop
Catálogo:           workshop_caroljf
Schema:             workshop
Prefixo:            caroljf
Volume:             workshop_caroljf.workshop.caroljf_volume
Volume Path:        /Volumes/workshop_caroljf/workshop/caroljf_volume

Tabelas:
  • workshop_caroljf.workshop.caroljf_contratos
  • workshop_caroljf.workshop.caroljf_fornecedores
  • workshop_caroljf.workshop.caroljf_ordens_servico

✅ Widgets criados — células SQL podem usar ${TABLE_CONTRATOS}, etc.


In [0]:
print(f"⏳ Criando volume {CATALOG}.{SCHEMA}.{VOLUME_NAME}...")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
print(f"✅ Volume criado: {FULL_VOLUME_PATH}")
print()

⏳ Criando volume workshop_caroljf.workshop.caroljf_volume...
✅ Volume criado: /Volumes/workshop_caroljf/workshop/caroljf_volume



In [0]:
import os, shutil

# Caminho dos CSVs no Workspace
CSV_BASE_PATH = os.path.join(os.path.dirname(os.path.realpath('__file__')), "quimica_dataset")


print(f"📂 CSVs origem: {CSV_BASE_PATH}")
print(f"📦 Volume destino: {FULL_VOLUME_PATH}")
print()

# -------------------------------------------------------
# 1. Copiar CSVs do Workspace para o Volume
#    (Volumes são acessíveis tanto por Python I/O quanto pelo Spark)
# -------------------------------------------------------
csv_dest = os.path.join(FULL_VOLUME_PATH, "csvs")
os.makedirs(csv_dest, exist_ok=True)

csv_files = [f for f in os.listdir(CSV_BASE_PATH) if f.endswith(".csv")]
for csv_file in csv_files:
    src = os.path.join(CSV_BASE_PATH, csv_file)
    dst = os.path.join(csv_dest, csv_file)
    shutil.copy2(src, dst)
    print(f"   ✅ {csv_file} copiado para o volume")

print(f"\n📁 CSVs disponíveis em: {csv_dest}")

# -------------------------------------------------------
# 2. Criar tabelas lendo os CSVs do Volume
#    (No Serverless, Spark consegue ler de /Volumes/ sem problemas)
# -------------------------------------------------------
csv_to_table = {
    "contratos.csv": TABLE_CONTRATOS,
    "fornecedores.csv": TABLE_FORNECEDORES,
    "ordens_servico.csv": TABLE_ORDENS_SERVICO,
}

print()
for csv_file, table_name in csv_to_table.items():
    volume_csv_path = os.path.join(csv_dest, csv_file)
    print(f"⏳ Criando tabela {table_name} ...")
    
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("encoding", "UTF-8")
          .csv(volume_csv_path)
    )
    
    df.write.mode("overwrite").saveAsTable(table_name)
    
    row_count = spark.table(table_name).count()
    print(f"   ✅ {table_name} — {row_count} registros")

print("\n🎉 Todas as tabelas foram criadas com sucesso!")

📂 CSVs origem: /Workspace/Users/carolina.ferreira@databricks.com/quimica_dataset
📦 Volume destino: /Volumes/workshop_caroljf/workshop/caroljf_volume

   ✅ fornecedores.csv copiado para o volume
   ✅ contratos.csv copiado para o volume
   ✅ ordens_servico.csv copiado para o volume

📁 CSVs disponíveis em: /Volumes/workshop_caroljf/workshop/caroljf_volume/csvs

⏳ Criando tabela workshop_caroljf.workshop.caroljf_contratos ...
   ✅ workshop_caroljf.workshop.caroljf_contratos — 1000 registros
⏳ Criando tabela workshop_caroljf.workshop.caroljf_fornecedores ...
   ✅ workshop_caroljf.workshop.caroljf_fornecedores — 100 registros
⏳ Criando tabela workshop_caroljf.workshop.caroljf_ordens_servico ...
   ✅ workshop_caroljf.workshop.caroljf_ordens_servico — 500 registros

🎉 Todas as tabelas foram criadas com sucesso!


In [0]:
# Verificação rápida das tabelas
for table in [TABLE_CONTRATOS, TABLE_FORNECEDORES, TABLE_ORDENS_SERVICO]:
    print(f"\n📊 {table}")
    display(spark.table(table).limit(3))


📊 workshop_caroljf.workshop.caroljf_contratos


id_contrato,numero_contrato,id_fornecedor,tipo_contrato,descricao_escopo,valor_total,moeda,data_inicio,data_fim,status,unidade_industrial,area_responsavel,documento_contrato,procedimento_referencia
CTR-00001,CT-2023-00001,FORN-0004,Contratação de Serviços de Inspeção,Manutenção preventiva e corretiva de trocadores de calor e torres de resfriamento,204986.18,BRL,2023-04-20,2024-04-19,Ativo,Unidade Industrial Várzea Arlindo,Operações de Planta,CONTRATO_CT-2023-00001.pdf,PROC-SMS-005_Manuseio_Produtos_Quimicos.pdf
CTR-00002,CT-2022-00002,FORN-0017,Serviços de Construção e Montagem,Manutenção de sistemas elétricos e subestações de média tensão,1.171465218E7,BRL,2022-09-02,2024-02-29,Ativo,Polo Petroquímico de Santa Grande,Engenharia de Processos,CONTRATO_CT-2022-00002.pdf,PROC-QLD-001_Controle_Qualidade_Resinas.pdf
CTR-00003,CT-2021-00003,FORN-0058,Contratação de Transporte e Logística,"Calibração de instrumentos de medição de vazão, pressão e temperatura",1965687.98,BRL,2021-09-09,2023-03-08,Ativo,Polo Petroquímico de Lagoa das Águas,Projetos e Expansão,CONTRATO_CT-2021-00003.pdf,PROC-SMS-001_Permissao_Trabalho_Quente.pdf



📊 workshop_caroljf.workshop.caroljf_fornecedores


id_fornecedor,razao_social,cnpj,categoria,cidade,estado,telefone,email,status_cadastro,data_cadastro,avaliacao_desempenho
FORN-0001,Abreu Manutencao,45.350.328/0001-27,Revestimentos e Isolamentos,Lagoa das Águas,BA,(13) 98935-2424,abreu.manutencao@empresa.com.br,Ativo,2018-05-11,5.1
FORN-0002,Abreu Manutencao do Brasil,74.716.127/0001-81,Serviços de Engenharia,Vila da Colina,RJ,(82) 96873-4611,abreu.manutencao.do.brasil@empresa.com.br,Ativo,2021-02-13,9.0
FORN-0003,Albuquerque Equipamentos,99.532.448/0001-45,Materiais de Consumo,Lagoa do Sul,SP,(21) 95514-2674,albuquerque.equipamentos@empresa.com.br,Ativo,2019-02-01,6.8



📊 workshop_caroljf.workshop.caroljf_ordens_servico


id_ordem,numero_ordem,id_contrato,id_fornecedor,tipo_ordem,descricao,valor_ordem,data_emissao,data_prevista_conclusao,data_conclusao,status,prioridade,unidade_industrial,area_solicitante
OS-00001,OS-2022-97320,CTR-00072,FORN-0009,Fornecimento,Atendimento emergencial - Projetos e Expansão,753482.57,2022-06-28,2022-08-12,null,Em Andamento,Crítica,Unidade Industrial Ribeirão das Águas,Projetos e Expansão
OS-00002,OS-2024-62412,CTR-00491,FORN-0099,Emergencial,Atendimento emergencial - Automação e Instrumentação,436444.27,2024-10-08,2024-11-22,null,Em Andamento,Média,Unidade Industrial Várzea Arlindo,Automação e Instrumentação
OS-00003,OS-2023-40999,CTR-00216,FORN-0098,Serviço,Fornecimento parcial conforme contrato CT-2023-00216,11009.62,2023-06-11,2023-06-18,2023-06-17,Concluída,Alta,Polo Petroquímico de Santa Grande,Manutenção Industrial


In [0]:
import shutil

# Pastas de PDFs para upload
WORKSPACE_BASE = os.path.join(os.path.dirname(os.path.realpath('__file__')))
pdf_folders = {
    "contratos": os.path.join(WORKSPACE_BASE, "pdfs_contratos"),
    "procedimentos": os.path.join(WORKSPACE_BASE, "pdfs_procedimentos"),
}

total_uploaded = 0

for subfolder, source_path in pdf_folders.items():
    # Criar subpasta no volume
    dest_dir = os.path.join(FULL_VOLUME_PATH, subfolder)
    os.makedirs(dest_dir, exist_ok=True)
    
    pdf_files = [f for f in os.listdir(source_path) if f.endswith('.pdf')]
    print(f"📁 Enviando {len(pdf_files)} PDFs para {dest_dir}")
    
    for pdf_file in sorted(pdf_files):
        src = os.path.join(source_path, pdf_file)
        dst = os.path.join(dest_dir, pdf_file)
        shutil.copy2(src, dst)
        total_uploaded += 1
    
    print(f"   ✅ {len(pdf_files)} arquivos copiados")

print(f"\n🎉 Upload concluído! {total_uploaded} PDFs no volume.")
print(f"\n📂 Estrutura do volume:")
for root, dirs, files in os.walk(FULL_VOLUME_PATH):
    level = root.replace(FULL_VOLUME_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for f in sorted(files)[:5]:
        print(f"{sub_indent}📄 {f}")
    if len(files) > 5:
        print(f"{sub_indent}... e mais {len(files) - 5} arquivos")

📁 Enviando 30 PDFs para /Volumes/workshop_caroljf/workshop/caroljf_volume/contratos
   ✅ 30 arquivos copiados
📁 Enviando 15 PDFs para /Volumes/workshop_caroljf/workshop/caroljf_volume/procedimentos
   ✅ 15 arquivos copiados

🎉 Upload concluído! 45 PDFs no volume.

📂 Estrutura do volume:
📁 caroljf_volume/
  📁 csvs/
    📄 contratos.csv
    📄 fornecedores.csv
    📄 ordens_servico.csv
  📁 contratos/
    📄 CONTRATO_CT-2020-00009.pdf
    📄 CONTRATO_CT-2020-00087.pdf
    📄 CONTRATO_CT-2020-00289.pdf
    📄 CONTRATO_CT-2020-00438.pdf
    📄 CONTRATO_CT-2020-00633.pdf
    ... e mais 25 arquivos
  📁 procedimentos/
    📄 PROC-AMB-001_Gestao_Residuos_Industriais.pdf
    📄 PROC-AMB-002_Monitoramento_Emissoes.pdf
    📄 PROC-ENG-001_Gestao_Mudancas_MOC.pdf
    📄 PROC-ENG-002_Comissionamento_Startup.pdf
    📄 PROC-LOG-001_Transporte_Produtos_Perigosos.pdf
    ... e mais 10 arquivos


---
# Módulo 1 — Genie (Dados Estruturados em Linguagem Natural)
> **Objetivo:** Criar um Genie Space que permita fazer perguntas sobre contratos, fornecedores e ordens de serviço usando linguagem natural.  
> **Pré-requisito:** Módulo 0 executado (tabelas criadas).

### Passo a Passo para Criar o Genie Space

1. **Acesse o menu lateral** → Clique em **"Genie"** (ou busque na barra de busca)
2. **Clique em "New"** para criar um novo Genie Space
3. **Configure o Genie Space:**
   - **Nome:** `Gestão Petroquímica - [seu prefixo]`
   - **Warehouse:** Selecione um SQL Warehouse disponível (ex: Serverless)
   - **Tabelas:** Adicione as 3 tabelas criadas no Módulo 0:
     - `{CATALOG}.{SCHEMA}.{PREFIX}_contratos`
     - `{CATALOG}.{SCHEMA}.{PREFIX}_fornecedores`
     - `{CATALOG}.{SCHEMA}.{PREFIX}_ordens_servico`
4. **General Instructions (Instruções Gerais):** Cole o texto da célula abaixo
5. **Clique em "Save"**

### Perguntas sugeridas para testar o Genie

**Básicas (validação):**
- *"Quantos contratos ativos temos?"*
- *"Liste os 10 maiores contratos por valor total"*
- *"Quantos fornecedores temos cadastrados por estado?"*

**Intermediárias (joins):**
- *"Quais fornecedores têm mais contratos ativos?"*
- *"Qual o valor total de contratos por unidade industrial?"*
- *"Quais ordens de serviço estão em andamento com prioridade crítica?"*

**Avançadas (análise):**
- *"Qual o valor médio de ordens de serviço por tipo, agrupado por unidade industrial?"*
- *"Quais fornecedores têm avaliação abaixo de 5 e possuem contratos ativos?"*
- *"Mostre a evolução mensal de ordens de serviço emitidas nos últimos 2 anos"*
- *"Quais contratos vencem nos próximos 90 dias e qual o fornecedor responsável?"*

**Modo Agente**

- *Por que alguns contratos ativos apresentam valores significativamente superiores à média do setor?*
- *Quais fatores explicam a concentração de ordens de serviço críticas em determinadas unidades industriais?*
- *Existe alguma relação entre a avaliação baixa de fornecedores e o aumento de penalidades contratuais? Explique*

In [0]:
# Execute esta célula para gerar as instruções a serem coladas no Genie Space

genie_instructions = f"""
Você é um assistente de dados de uma empresa petroquímica brasileira que produz resinas termoplásticas.

Contexto do negócio:
- A empresa gerencia contratos com fornecedores de serviços industriais (manutenção, engenharia, logística, etc.)
- Existem ordens de serviço (OS) vinculadas a contratos e fornecedores
- As unidades industriais são plantas petroquímicas espalhadas pelo Brasil

Tabelas disponíveis:
1. {TABLE_CONTRATOS} - Contratos com fornecedores (valor, status, datas, escopo)
2. {TABLE_FORNECEDORES} - Cadastro de fornecedores (razão social, CNPJ, categoria, avaliação)
3. {TABLE_ORDENS_SERVICO} - Ordens de serviço operacionais (tipo, valor, prioridade, status)

Relacionamentos:
- contratos.id_fornecedor = fornecedores.id_fornecedor
- ordens_servico.id_contrato = contratos.id_contrato
- ordens_servico.id_fornecedor = fornecedores.id_fornecedor

Regras:
- Valores monetários estão em BRL (Reais)
- Datas estão no formato YYYY-MM-DD
- Sempre formate valores com R$ e separador de milhar
- Ao listar fornecedores, inclua a razão social e o CNPJ
- Prioridades de OS: Crítica, Alta, Média, Baixa
- Status de contratos: Ativo, Encerrado, Suspenso
"""

print(genie_instructions)
print("\n" + "="*60)


Você é um assistente de dados de uma empresa petroquímica brasileira que produz resinas termoplásticas.

Contexto do negócio:
- A empresa gerencia contratos com fornecedores de serviços industriais (manutenção, engenharia, logística, etc.)
- Existem ordens de serviço (OS) vinculadas a contratos e fornecedores
- As unidades industriais são plantas petroquímicas espalhadas pelo Brasil

Tabelas disponíveis:
1. workshop_caroljf.workshop.caroljf_contratos - Contratos com fornecedores (valor, status, datas, escopo)
2. workshop_caroljf.workshop.caroljf_fornecedores - Cadastro de fornecedores (razão social, CNPJ, categoria, avaliação)
3. workshop_caroljf.workshop.caroljf_ordens_servico - Ordens de serviço operacionais (tipo, valor, prioridade, status)

Relacionamentos:
- contratos.id_fornecedor = fornecedores.id_fornecedor
- ordens_servico.id_contrato = contratos.id_contrato
- ordens_servico.id_fornecedor = fornecedores.id_fornecedor

Regras:
- Valores monetários estão em BRL (Reais)
- Datas 

---
# Módulo 2 — AI Functions *(Opcional)*
> **Objetivo:** Demonstrar como usar funções de IA diretamente no SQL para enriquecer e analisar dados.  
> **Pré-requisito:** Módulo 0 executado.

As **AI Functions** do Databricks permitem chamar modelos de linguagem diretamente em queries SQL para:
- Classificar dados
- Extrair informações
- Gerar resumos
- Analisar sentimento

In [0]:
%sql
-- Exemplo: Usar IA para classificar a criticidade real de ordens de serviço
-- baseado na descrição textual
SELECT 
  numero_ordem,
  descricao,
  prioridade as prioridade_original,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Classifique a seguinte descrição de ordem de serviço petroquímica em: CRITICA, ALTA, MEDIA ou BAIXA. ',
      'Responda APENAS com a classificação, sem explicação. Descrição: ', descricao
    )
  ) as prioridade_ia
FROM ${TABLE_ORDENS_SERVICO} LIMIT 10

numero_ordem,descricao,prioridade_original,prioridade_ia
OS-2022-97320,Atendimento emergencial - Projetos e Expansão,Crítica,CRITICA
OS-2024-62412,Atendimento emergencial - Automação e Instrumentação,Média,CRITICA
OS-2023-40999,Fornecimento parcial conforme contrato CT-2023-00216,Alta,BAIXA
OS-2024-90951,Medição mensal #4 - Contratação de Serviços de Inspeção,Média,BAIXA
OS-2024-66250,Atendimento emergencial - Logística e Suprimentos,Crítica,CRITICA
OS-2021-92432,Mobilização de equipe para contratação de serviços de manutenção na Polo Petroquímico de Santa Grande,Baixa,MEDIA
OS-2023-90801,Execução de serviços de andaime e acesso para paradas programadas de man - Etapa 4,Baixa,MÉDIA
OS-2023-65846,Execução de projeto e instalação de sistema de automação para planta de - Etapa 2,Alta,ALTA
OS-2025-64797,Fornecimento parcial conforme contrato CT-2024-00768,Baixa,BAIXA
OS-2020-89498,Atendimento emergencial - Utilidades e Energia,Média,CRITICA


In [0]:
%sql
-- Exemplo: Gerar resumo executivo dos maiores contratos
SELECT 
  numero_contrato,
  valor_total,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Resuma em uma frase curta (máx 15 palavras) o escopo deste contrato petroquímico: ', 
      descricao_escopo
    )
  ) as resumo_escopo
FROM ${TABLE_CONTRATOS}
LIMIT 10

numero_contrato,valor_total,resumo_escopo
CT-2023-00001,204986.18,Manutenção de equipamentos petroquímicos.
CT-2022-00002,1.171465218E7,Manutenção de sistemas elétricos e subestações de média tensão.
CT-2021-00003,1965687.98,Calibração de instrumentos de medição em instalações petroquímicas.
CT-2023-00004,1287662.59,Fornecimento de peças para equipamentos rotativos petroquímicos.
CT-2023-00005,4716088.33,Inspeção de vasos e tubulações por métodos não destrutivos.
CT-2022-00006,3487178.66,Manutenção de sistemas elétricos e subestações de média tensão.
CT-2022-00007,1300059.36,Fornecimento de gases industriais para uso petroquímico.
CT-2022-00008,1.004359637E7,Fornecimento de gases industriais para uso petroquímico.
CT-2020-00009,844790.31,Inspeção de vasos e tubulações por ensaios não destrutivos.
CT-2022-00010,2633461.38,Fornecimento de válvulas para unidades de polimerização.


In [0]:
%sql
-- Exemplo: Avaliar risco de fornecedores com base em múltiplos fatores
SELECT 
  f.razao_social,
  f.avaliacao_desempenho,
  f.status_cadastro,
  COUNT(c.id_contrato) as total_contratos,
  SUM(c.valor_total) as valor_total_contratos,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Avalie o nível de risco (BAIXO, MEDIO, ALTO) deste fornecedor petroquímico considerando: ',
      'Avaliação: ', CAST(f.avaliacao_desempenho AS STRING), '/10, ',
      'Status: ', f.status_cadastro, ', ',
      'Total de contratos: ', CAST(COUNT(c.id_contrato) AS STRING), ', ',
      'Valor total: R$ ', CAST(ROUND(SUM(c.valor_total), 2) AS STRING), '. ',
      'Responda APENAS com: BAIXO, MEDIO ou ALTO'
    )
  ) as risco_ia
FROM ${TABLE_FORNECEDORES} f
LEFT JOIN ${TABLE_CONTRATOS} c ON f.id_fornecedor = c.id_fornecedor
GROUP BY f.id_fornecedor, f.razao_social, f.avaliacao_desempenho, f.status_cadastro
ORDER BY f.avaliacao_desempenho ASC
LIMIT 2

razao_social,avaliacao_desempenho,status_cadastro,total_contratos,valor_total_contratos,risco_ia
Araújo Moraes Automacao,5.0,Ativo,7,2.615717496E7,MEDIO
Carvalho Instrumentacao,5.0,Inativo,11,4.577542267999999E7,ALTO


   
---
# Módulo 3 — Unity Catalog Functions *(Opcional)*
> **Objetivo:** Criar funções SQL reutilizáveis no Unity Catalog que podem ser usadas pelo Genie e como tools de agentes no AgentBricks.  
> **Pré-requisito:** Módulo 0 executado.

### Por que usar UC Functions?
- **Genie:** Permite que o Genie execute lógica de negócio complexa sem precisar gerar SQL elaborado
- **AgentBricks:** Funções podem ser adicionadas como **tools** de agentes, dando-lhes capacidade de consultar dados estruturados
- **Reuso:** Uma vez criada, a função fica disponível para qualquer usuário com permissão

### Funções que vamos criar:
| Função | Tipo | Recebe | Retorna | Uso |
|--------|------|--------|---------|-----|
| `resumo_fornecedor` | **Tabular** | Nome (texto livre) | Tabela com resumo | `SELECT * FROM func('Abreu')` |
| `valor_total_fornecedor` | **Escalar** | ID do fornecedor | DOUBLE (R$) | `SELECT func('FORN-0004')` |
| `qtd_os_abertas` | **Escalar** | ID do contrato | INT (quantidade) | Inline: `SELECT func(coluna) FROM tabela` |

In [0]:
%sql
-- Função que retorna um resumo dos contratos de um fornecedor
CREATE OR REPLACE FUNCTION ${CATALOG}.${SCHEMA}.${PREFIX}_resumo_fornecedor(p_razao_social STRING)
RETURNS TABLE (
  razao_social STRING,
  total_contratos INT,
  valor_total_contratos DOUBLE,
  contratos_ativos INT,
  categorias_contrato STRING
)
RETURN
  SELECT 
    f.razao_social,
    COUNT(c.id_contrato) as total_contratos,
    ROUND(SUM(c.valor_total), 2) as valor_total_contratos,
    SUM(CASE WHEN c.status = 'Ativo' THEN 1 ELSE 0 END) as contratos_ativos,
    CONCAT_WS(', ', COLLECT_SET(c.tipo_contrato)) as categorias_contrato
  FROM ${TABLE_FORNECEDORES} f
  LEFT JOIN ${TABLE_CONTRATOS} c ON f.id_fornecedor = c.id_fornecedor
  WHERE LOWER(f.razao_social) LIKE CONCAT('%', LOWER(p_razao_social), '%')
  GROUP BY f.razao_social

In [0]:
%sql
    
-- Função ESCALAR: retorna um único valor (DOUBLE)
-- Útil para cálculos pontuais e como tool de agentes
CREATE OR REPLACE FUNCTION ${CATALOG}.${SCHEMA}.${PREFIX}_valor_total_fornecedor(
  p_id_fornecedor STRING
  COMMENT 'ID do fornecedor no formato FORN-XXXX'
)
RETURNS DOUBLE
COMMENT 'Retorna o valor total (R$) de contratos ativos de um fornecedor'
RETURN
  SELECT COALESCE(ROUND(SUM(c.valor_total), 2), 0.0)
  FROM ${TABLE_CONTRATOS} c
  WHERE c.id_fornecedor = p_id_fornecedor
    AND c.status = 'Ativo'

In [0]:
%sql
-- Função ESCALAR que recebe o valor de uma coluna e retorna um número
-- Pode ser chamada INLINE no SELECT, aplicada a cada linha da tabela
CREATE OR REPLACE FUNCTION ${CATALOG}.${SCHEMA}.${PREFIX}_qtd_os_abertas(
  p_id_contrato STRING
  COMMENT 'ID do contrato no formato CTR-XXXXX'
)
RETURNS INT
COMMENT 'Retorna a quantidade de ordens de serviço não concluídas de um contrato'
RETURN
  SELECT COUNT(*)
  FROM ${TABLE_ORDENS_SERVICO} os
  WHERE os.id_contrato = p_id_contrato
    AND os.status != 'Concluída'

In [0]:
%sql
-- =============================================
-- Testar FUNÇÃO TABULAR (retorna múltiplas linhas)
-- Chamada obrigatória com SELECT * FROM
-- =============================================
SELECT * FROM ${CATALOG}.${SCHEMA}.${PREFIX}_resumo_fornecedor('Abreu');


razao_social,total_contratos,valor_total_contratos,contratos_ativos,categorias_contrato
Abreu Manutencao do Brasil,8,3.285774593E7,4,"Contratação de Serviços de Inspeção, Serviços de Engenharia e Projetos, Serviços de Automação e Instrumentação, Contratação de Transporte e Logística"
Abreu Manutencao,9,4.224493365E7,6,"Contratação de Transporte e Logística, Contratação de Serviços Ambientais, Serviços de Construção e Montagem, Fornecimento de Materiais e Equipamentos, Contratação de Serviços de Manutenção, Serviços de Calibração e Ensaios"
Marques Abreu,14,5.70258916E7,7,"Fornecimento de Materiais e Equipamentos, Contratação de Serviços de Manutenção, Contratação de Serviços Ambientais, Fornecimento de Produtos Químicos, Contratação de Serviços de Inspeção, Serviços de Calibração e Ensaios, Contratação de Transporte e Logística, Serviços de Engenharia e Projetos"


In [0]:
%sql

-- =============================================
-- Testar FUNÇÃO ESCALAR inline com coluna da tabela
-- A função recebe id_contrato de CADA LINHA e retorna a qtd de OS abertas
-- =============================================
SELECT 
  numero_contrato,
  status,
  valor_total,
  ${CATALOG}.${SCHEMA}.${PREFIX}_qtd_os_abertas(id_contrato) as os_abertas
FROM ${TABLE_CONTRATOS}
WHERE status = 'Ativo'
LIMIT 10;

numero_contrato,status,valor_total,os_abertas
CT-2023-00001,Ativo,204986.18,0
CT-2022-00002,Ativo,1.171465218E7,0
CT-2021-00003,Ativo,1965687.98,0
CT-2022-00006,Ativo,3487178.66,0
CT-2022-00008,Ativo,1.004359637E7,0
CT-2025-00013,Ativo,6649329.46,1
CT-2024-00016,Ativo,4802092.28,1
CT-2023-00019,Ativo,140881.72,2
CT-2020-00023,Ativo,508677.07,1
CT-2021-00026,Ativo,4793205.58,1


   
### 💡 Diferença entre função Tabular e Escalar

| Tipo | Retorno | Como chamar | Exemplo |
|------|---------|-------------|---------|
| **Tabular** (`RETURNS TABLE`) | Múltiplas linhas e colunas | `SELECT * FROM func(...)` | `resumo_fornecedor` |
| **Escalar** (`RETURNS DOUBLE`) | Um único valor | `SELECT func(...)` (inline) | `valor_total_fornecedor` |

### Como adicionar as UC Functions ao Genie Space

1. Abra seu Genie Space criado no Módulo 1
2. Clique no ícone de **configuração** (⚙️)
3. Na seção **"Tools"** ou **"Functions"**, clique em **"Add"**
4. Busque e adicione as funções:
   - `{CATALOG}.{SCHEMA}.{PREFIX}_resumo_fornecedor` (tabular)
   - `{CATALOG}.{SCHEMA}.{PREFIX}_valor_total_fornecedor` (escalar)
5. Salve as alterações

### Perguntas para testar com as funções no Genie:
- *"Mostre o resumo do fornecedor Abreu"* → usa a função tabular
- *"Qual o valor total de contratos ativos do fornecedor FORN-0004?"* → usa a função escalar

### Uso como Tool no AgentBricks:
Ambos os tipos podem ser adicionados como **tools** de agentes. O LLM decide quando chamar cada função com base na pergunta do usuário.

---
# Módulo 4 — AgentBricks: Knowledge Assistant com PDFs
> **Objetivo:** Criar um agente que responde perguntas sobre os documentos não estruturados (PDFs de contratos e procedimentos operacionais).  
> **Pré-requisito:** Módulo 0 executado (volume com PDFs criado).

### O que é o Knowledge Assistant?
O **Knowledge Assistant** do AgentBricks é um template de agente pré-configurado que:
- Indexa documentos (PDFs, DOCX, TXT) automaticamente usando **Vector Search**
- Cria um pipeline de **RAG (Retrieval-Augmented Generation)** sem código
- Permite perguntas em linguagem natural sobre o conteúdo dos documentos

---

### Passo a Passo para Criar o Knowledge Assistant

#### 1. Acessar o AgentBricks
- No menu lateral, clique em **"New"** → **"Agent"**
- Ou acesse diretamente: **Workspace** → **Mosaic AI** → **Agent Builder**

#### 2. Escolher o Template
- Selecione **"Knowledge Assistant"** entre os templates disponíveis
- Clique em **"Create"**

#### 3. Configurar o Agente
- **Nome do Agente:** `knowledge_assistant_petroquimica_[seu_prefixo]`
- **Instruções:** Cole o texto gerado na célula 4.1 abaixo

#### 4. Adicionar os Documentos (Data Sources)
- Na seção **"Data Sources"** ou **"Knowledge Base"**
- Clique em **"Add Data Source"** → **"Unity Catalog Volume"**
- Navegue até o volume: `{CATALOG}.{SCHEMA}.{VOLUME_NAME}`
- Ou insira o path diretamente (veja célula 4.2)
- O sistema irá automaticamente indexar os PDFs
- Adicione os dois datasources contratos e procedimentos

#### 5. Testar o Agente
- Use o painel de **chat de teste** para fazer perguntas
- Veja as perguntas sugeridas na célula 4.3

#### 6. Deploy (Opcional)
- Clique em **"Deploy"** para criar um endpoint de serving
- Isso será útil no Módulo 5 para integração

In [0]:
# System Prompt sugerido para o Knowledge Assistant

knowledge_assistant_prompt = f"""
Você é um assistente especializado em documentos operacionais de uma empresa petroquímica brasileira que produz resinas termoplásticas (PE, PP, PVC).

Sua base de conhecimento contém dois tipos de documentos:

1. **Contratos** (pasta contratos/): Documentos contratuais com fornecedores de serviços industriais. Contêm informações sobre:
   - Partes envolvidas (contratante e contratada)
   - Objeto e escopo do contrato
   - Valores, prazos e condições de pagamento
   - Cláusulas de rescisão, multas e penalidades
   - Requisitos de SMS (Saúde, Meio Ambiente e Segurança)
   - Obrigações trabalhistas

2. **Procedimentos Operacionais** (pasta procedimentos/): Documentos técnicos que cobrem:
   - SMS: Permissão de Trabalho Quente, Bloqueio/Etiquetagem (LOTO), Trabalho em Altura, Espaço Confinado, Manuseio de Produtos Químicos
   - Manutenção: Plano de Manutenção Preventiva, Parada Programada, Gestão de Sobressalentes
   - Qualidade: Controle de Qualidade de Resinas, Especificação Técnica de Materiais
   - Engenharia: Gestão de Mudanças (MOC), Comissionamento e Startup
   - Ambiental: Gestão de Resíduos Industriais, Monitoramento de Emissões
   - Logística: Transporte de Produtos Perigosos

Regras:
- Sempre cite o documento fonte ao responder
- Se a informação não estiver nos documentos, diga claramente
- Responda em português brasileiro
- Use terminologia técnica da indústria petroquímica
"""

print(knowledge_assistant_prompt)
print("\n" + "="*60)



Você é um assistente especializado em documentos operacionais de uma empresa petroquímica brasileira que produz resinas termoplásticas (PE, PP, PVC).

Sua base de conhecimento contém dois tipos de documentos:

1. **Contratos** (pasta contratos/): Documentos contratuais com fornecedores de serviços industriais. Contêm informações sobre:
   - Partes envolvidas (contratante e contratada)
   - Objeto e escopo do contrato
   - Valores, prazos e condições de pagamento
   - Cláusulas de rescisão, multas e penalidades
   - Requisitos de SMS (Saúde, Meio Ambiente e Segurança)
   - Obrigações trabalhistas

2. **Procedimentos Operacionais** (pasta procedimentos/): Documentos técnicos que cobrem:
   - SMS: Permissão de Trabalho Quente, Bloqueio/Etiquetagem (LOTO), Trabalho em Altura, Espaço Confinado, Manuseio de Produtos Químicos
   - Manutenção: Plano de Manutenção Preventiva, Parada Programada, Gestão de Sobressalentes
   - Qualidade: Controle de Qualidade de Resinas, Especificação Técnica de 

In [0]:
# Path do volume para configurar no Knowledge Assistant
print("\n📎 Informações para configurar o Data Source no Knowledge Assistant:")
print("=" * 60)
print(f"Volume UC:      {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
print(f"Volume Path:    {FULL_VOLUME_PATH}")
print(f"")
print(f"Subpastas disponíveis:")
print(f"  📁 {FULL_VOLUME_PATH}/contratos/     (30 PDFs de contratos)")
print(f"  📁 {FULL_VOLUME_PATH}/procedimentos/  (15 PDFs de procedimentos)")
print(f"")
print(f"💡 Dica: adicione o volume raiz para indexar todos os PDFs de uma vez")


📎 Informações para configurar o Data Source no Knowledge Assistant:
Volume UC:      workshop_caroljf.workshop.caroljf_volume
Volume Path:    /Volumes/workshop_caroljf/workshop/caroljf_volume

Subpastas disponíveis:
  📁 /Volumes/workshop_caroljf/workshop/caroljf_volume/contratos/     (30 PDFs de contratos)
  📁 /Volumes/workshop_caroljf/workshop/caroljf_volume/procedimentos/  (15 PDFs de procedimentos)

💡 Dica: adicione o volume raiz para indexar todos os PDFs de uma vez


### 🧠 Perguntas sugeridas para testar o Knowledge Assistant

**Sobre Procedimentos de SMS:**
- *"Quais são os passos para solicitar uma Permissão de Trabalho Quente?"*
- *"O que é o procedimento LOTO e quando deve ser aplicado?"*
- *"Quais EPIs são obrigatórios para trabalho em espaço confinado?"*
- *"Quais são os requisitos para manuseio de produtos químicos na planta?"*

**Sobre Manutenção e Engenharia:**
- *"Qual a frequência recomendada de manutenção preventiva para equipamentos rotativos?"*
- *"Quais são as etapas de uma parada programada?"*
- *"O que é o processo de MOC (Gestão de Mudanças)?"*
- *"Quais são os critérios de aceitação no controle de qualidade de resinas?"*

**Sobre Contratos:**
- *"Quais são as cláusulas de penalidade do contrato CT-2023-00291?"*
- *"Qual o escopo do contrato CT-2024-00582?"*
- *"Quais contratos mencionam requisitos de segurança do trabalho?"*
- *"Resuma as obrigações trabalhistas do contrato CT-2025-00968"*

**Sobre Meio Ambiente e Logística:**
- *"Como é feito o monitoramento de emissões atmosféricas?"*
- *"Quais são as classes de resíduos industriais e como devem ser gerenciados?"*
- *"Quais documentações são necessárias para transporte de produtos perigosos?"*

---
# Módulo 5 — Agent Supervisor: Orquestrando Dados Estruturados + Não Estruturados 🌟
> **Objetivo:** Criar um **Agent Supervisor** no AgentBricks que orquestra dois sub-agentes especializados — um para dados estruturados (Genie) e outro para documentos (Knowledge Assistant) — decidindo automaticamente qual(is) consultar.  
> **Pré-requisito:** Módulo 1 (Genie Space criado) e Módulo 4 (Knowledge Assistant criado e publicado como endpoint).

---

### Arquitetura do Agent Supervisor

```
                    ┌─────────────────────────┐
                    │   🧑‍💻 Usuário faz uma    │
                    │      pergunta            │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌─────────────────────────┐
                    │  🧠 AGENT SUPERVISOR     │
                    │  Analisa a pergunta e    │
                    │  decide a rota           │
                    └──────┬──────────┬────────┘
                           │          │
              ┌────────────▼──┐  ┌────▼────────────┐
              │ 📊 Genie      │  │ 📄 Knowledge     │
              │ (Estruturado) │  │ Assistant        │
              │               │  │ (Não Estruturado)│
              │ Tabelas:      │  │ PDFs:            │
              │ • Contratos   │  │ • Contratos      │
              │ • Fornecedores│  │ • Procedimentos  │
              │ • Ordens Serv.│  │   Operacionais   │
              └───────────────┘  └──────────────────┘
```

### Por que usar um Supervisor?
- **Roteamento inteligente:** O supervisor decide automaticamente se a pergunta precisa do Genie, do Knowledge Assistant, ou de **ambos**
- **Respostas combinadas:** Quando a pergunta cruza os dois mundos, o supervisor consulta ambos e **consolida** a resposta
- **Escalável:** É fácil adicionar novos sub-agentes depois (ex: agente de anomalias, agente de previsão)

---

### Passo a Passo

#### Etapa 1 — Garantir que os sub-agentes estejam prontos

| Sub-agente | Criado no | Status necessário |
|------------|-----------|------------------|
| **Genie Space** | Módulo 1 | Criado e funcional (testado com perguntas) |
| **Knowledge Assistant** | Módulo 4 | Criado, indexação dos PDFs completa |

> 💡 **Importante:** Os agentes não precisam estar publicados (deploy) para serem usados como sub-agentes do Supervisor.

#### Etapa 2 — Criar o Agent Supervisor

1. No menu lateral, clique em **"New"** → **"Agent"**
2. Selecione o template **"Agent Supervisor"** (ou "Multi-Agent")
3. Configure:
   - **Nome:** `supervisor_petroquimica_[seu_prefixo]`
   - **Instruções (System Prompt):** Cole o texto gerado na **célula 5.1**

#### Etapa 3 — Adicionar o Genie como sub-agente

1. Na seção **"Tools"** do Supervisor, clique em **"+ Add Tool"**
2. Selecione **"Genie"**
3. Busque o Genie Space criado no Módulo 1
4. O Genie será adicionado como ferramenta — o supervisor poderá fazer perguntas SQL através dele

#### Etapa 4 — Adicionar o Knowledge Assistant como sub-agente

1. Clique novamente em **"+ Add Tool"**
2. Selecione **"AI Agent"**
3. Busque o Knowledge Assistant criado no Módulo 4
4. Pronto — agora o supervisor pode consultar os PDFs através do Knowledge Assistant

#### Etapa 5 — (Opcional) Adicionar UC Functions como tools adicionais

Se executou o Módulo 3, adicione também:
1. **"+ Add Tool"** → **"Unity Catalog Function"**
2. Adicione:
   - `resumo_fornecedor` (tabular)
   - `valor_total_fornecedor` (escalar)
   - `qtd_os_abertas` (escalar)

#### Etapa 6 — Testar o Supervisor

Use o painel de **chat** para testar com as perguntas da **célula 5.2**. Observe:
- O supervisor mostra **qual tool** está acionando a cada passo
- Para perguntas cruzadas, ele aciona **múltiplas tools** sequencialmente
- A resposta final **consolida** informações de ambas as fontes

In [0]:
# System Prompt para o Agent Supervisor

supervisor_prompt = f"""
Você é o agente supervisor de uma empresa petroquímica brasileira que produz resinas termoplásticas (PE, PP, PVC).

Seu papel é ORQUESTRAR dois sub-agentes especializados para responder perguntas dos usuários.

## Sub-agentes disponíveis:

### 1. Genie (Dados Estruturados)
Use para perguntas sobre:
- Valores, datas, status e quantidades de CONTRATOS
- Cadastro de FORNECEDORES (razão social, CNPJ, avaliação, categoria)
- ORDENS DE SERVIÇO (tipo, prioridade, valor, status, datas)
- Agregações numéricas (totais, médias, contagens, rankings)
- Cruzamentos entre tabelas (ex: fornecedores com mais contratos)

### 2. Knowledge Assistant (Documentos Não Estruturados)
Use para perguntas sobre:
- CLÁUSULAS, termos e penalidades de contratos específicos
- PROCEDIMENTOS OPERACIONAIS (SMS, manutenção, qualidade, engenharia)
- Normas de segurança (LOTO, trabalho em altura, espaço confinado)
- Especificações técnicas e requisitos ambientais
- Conteúdo textual detalhado dos documentos PDF

## Regras de roteamento:

1. **Pergunta sobre DADOS NUMÉRICOS ou LISTAS** → acione o Genie
2. **Pergunta sobre CONTEÚDO DE DOCUMENTOS** → acione o Knowledge Assistant
3. **Pergunta que CRUZA ambos os mundos** → acione AMBOS:
   - Primeiro o Genie para obter dados estruturados
   - Depois o Knowledge Assistant para complementar com detalhes dos documentos
   - Consolide as respostas em uma única resposta coerente

4. Sempre indique de qual fonte veio cada informação: 📊 (Genie) ou 📄 (Knowledge Assistant)
5. Responda em português brasileiro
6. Use terminologia técnica da indústria petroquímica
7. Na dúvida sobre qual sub-agente usar, consulte ambos

## Exemplo de roteamento:
- "Qual o valor do contrato CT-2023-00291?" → Genie
- "Quais as cláusulas de penalidade do CT-2023-00291?" → Knowledge Assistant
- "Qual o valor e as cláusulas de penalidade do CT-2023-00291?" → Genie + Knowledge Assistant
"""

print(supervisor_prompt)
print("\n" + "="*60)



Você é o agente supervisor de uma empresa petroquímica brasileira que produz resinas termoplásticas (PE, PP, PVC).

Seu papel é ORQUESTRAR dois sub-agentes especializados para responder perguntas dos usuários.

## Sub-agentes disponíveis:

### 1. Genie (Dados Estruturados)
Use para perguntas sobre:
- Valores, datas, status e quantidades de CONTRATOS
- Cadastro de FORNECEDORES (razão social, CNPJ, avaliação, categoria)
- ORDENS DE SERVIÇO (tipo, prioridade, valor, status, datas)
- Agregações numéricas (totais, médias, contagens, rankings)
- Cruzamentos entre tabelas (ex: fornecedores com mais contratos)

### 2. Knowledge Assistant (Documentos Não Estruturados)
Use para perguntas sobre:
- CLÁUSULAS, termos e penalidades de contratos específicos
- PROCEDIMENTOS OPERACIONAIS (SMS, manutenção, qualidade, engenharia)
- Normas de segurança (LOTO, trabalho em altura, espaço confinado)
- Especificações técnicas e requisitos ambientais
- Conteúdo textual detalhado dos documentos PDF

## Regras 

### 🧪 Perguntas para Testar o Agent Supervisor

As perguntas estão organizadas por tipo de roteamento — observe no chat qual sub-agente o supervisor aciona.

---

#### 📊 Rota: Apenas Genie (dados estruturados)
O supervisor deve acionar **somente o Genie**:
- *"Quantos contratos ativos temos e qual o valor total?"*
- *"Quais são os 5 fornecedores com pior avaliação de desempenho?"*
- *"Quantas ordens de serviço emergenciais estão em andamento?"*

#### 📄 Rota: Apenas Knowledge Assistant (documentos PDF)
O supervisor deve acionar **somente o Knowledge Assistant**:
- *"Quais são os passos para solicitar uma Permissão de Trabalho Quente?"*
- *"O que diz o procedimento de Bloqueio e Etiquetagem (LOTO)?"*
- *"Quais são os critérios de aceitação no controle de qualidade de resinas?"*

#### 🔀 Rota: Ambos (cruzamento estruturado + não estruturado)
O supervisor deve acionar **Genie + Knowledge Assistant** e consolidar:

**Nível 1 — Contrato + Documento:**
- *"Qual o valor do contrato CT-2023-00291 e quais são suas cláusulas de penalidade?"*
- *"O contrato CT-2024-00582 menciona requisitos de LOTO? E qual seu status atual na base?"*
- *"Liste os 3 maiores contratos ativos e resuma o escopo detalhado de cada um"*

**Nível 2 — Fornecedor + Procedimento:**
- *"Quais fornecedores atendem a unidade de Santa Grande e quais procedimentos de SMS se aplicam?"*
- *"O fornecedor do contrato CT-2025-00968 precisa seguir procedimento de trabalho em altura? Qual a avaliação dele?"*

**Nível 3 — Análise completa (múltiplas fontes):**
- *"Preciso planejar uma parada programada na unidade Ribeirão das Águas. Quais contratos estão ativos, quais OS estão pendentes, e qual o procedimento de parada?"*
- *"Faça uma análise de risco: fornecedores com avaliação abaixo de 6 que têm contratos ativos — os contratos deles têm cláusulas de segurança?"*
- *"Resuma a situação completa do contrato CT-2022-00693: valor, fornecedor, avaliação, OS vinculadas e cláusulas principais do PDF"*

### 🎯 Dicas para a Demo do Supervisor

#### O que observar durante o teste:
1. **Trace do roteamento:** No painel de chat, expanda os detalhes de cada resposta para ver:
   - Qual tool/sub-agente foi acionado
   - O prompt enviado para cada sub-agente
   - O tempo de resposta de cada um
2. **Consolidação:** Nas perguntas cruzadas, observe como o supervisor **combina** as respostas do Genie e do Knowledge Assistant em uma resposta única e coerente
3. **Indicação de fonte:** O prompt instrui o supervisor a marcar com 📊 (Genie) e 📄 (Knowledge Assistant) de onde veio cada informação

#### Sequência sugerida para a demo ao vivo:
| Ordem | Pergunta | Rota esperada | O que demonstra |
|-------|----------|---------------|-----------------|
| 1ª | *"Quantos contratos ativos temos?"* | Genie | Supervisor sabe rotear para dados estruturados |
| 2ª | *"O que é o procedimento LOTO?"* | Knowledge Assistant | Supervisor sabe rotear para documentos |
| 3ª | *"Qual o valor do CT-2023-00291 e suas cláusulas de penalidade?"* | **Ambos** | O grande momento: cruzamento dos dois mundos |
| 4ª | *"Planeje uma parada na unidade Ribeirão das Águas..."* | **Ambos** | Análise complexa com múltiplas fontes |

#### Troubleshooting:
- Se o supervisor não acionar o sub-agente correto, **refine as instruções** (System Prompt) com exemplos mais específicos
- Se a resposta consolidada não for boa, adicione na instrução: *"Sempre apresente a resposta em seções claras separando dados numéricos de conteúdo documental"*
- Para perguntas ambíguas, o supervisor deve consultar ambos — se não o fizer, adicione: *"Na dúvida, consulte ambos os sub-agentes"*



# Módulo 6: Databricks Apps e Genie Code

## 1. Criar um aplicativo Hello World com Streamlit 

Crie um Databricks Apps a partir do Streamlit.


## 2. Modificar o aplicativo com Genie Code

Digite algum prompt no Genie Code para modificar o app do Streamlit. Algo que seja simples tipo o background ou um botão.



# Módulo 7: AI/BI + Genie

Vamos criar um dashboard no AI/BI e fazer perguntas.

Alguns prompts para criação de gráficos:

* Crie um gráfico que mostre o número de orders de serviço por unidade industrial
* Popule os gráficos e estatístiscas em duas abas uma sobre contratos e outra sobre ordens de serviço. Inclua métricas sobre os fornecedores.




